<a href="https://colab.research.google.com/github/Farhad-Norouzzadeh/LSTM_implementation_python/blob/main/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Long Short-Term Memory (LSTM) — From Scratch in PyTorch

## 1. Introduction

Long Short-Term Memory (LSTM) is a type of Recurrent Neural Network (RNN) designed to learn dependencies across long sequences.

Traditional RNNs often struggle with **vanishing and exploding gradients** during Backpropagation Through Time (BPTT).

- **Vanishing gradients:** Gradients become extremely small as they propagate backward through time, making long-term dependencies difficult to learn.
- **Exploding gradients:** Gradients become excessively large, causing unstable weight updates and potentially preventing convergence.

LSTMs address these challenges by introducing a **cell state** and a set of learnable gates that control the flow of information.

Unlike traditional RNNs, which maintain one main hidden state, LSTMs maintain two states:

- **Cell state ($c_t$):** Carries long-term information across time steps.
- **Hidden state ($h_t$):** Represents the currently exposed information and can be used for predictions.

The cell state provides a more direct path for gradient propagation, helping reduce the vanishing-gradient problem.

---

## 2. LSTM Architecture

At time step $t$, an LSTM receives:

$$
x_t,\quad h_{t-1},\quad c_{t-1}
$$

Where:

- $x_t$: Current input vector.
- $h_{t-1}$: Hidden state from the previous time step.
- $c_{t-1}$: Cell state from the previous time step.

The LSTM produces:

$$
h_t,\quad c_t
$$

The information flow is:

```text
Current input (x_t) ───────────┐
                               ├──► Concatenation (z_t)
Previous hidden state (h_t-1) ─┘           │
                                          │
                  ┌───────────────────────┼───────────────────┐
                  │                       │                   │
                  ▼                       ▼                   ▼
            Forget gate             Input gate          Output gate
                (f_t)                  (i_t)                (o_t)
                  │                       │                   │
                  ▼                       ▼                   │
          Previous memory         Candidate memory           │
              (c_t-1)                 (g_t)                  │
                  │                       │                   │
                  └──────────┬────────────┘                   │
                             │                                │
                             ▼                                │
                      New cell state (c_t)                    │
                             │                                │
                             ▼                                │
                           tanh                               │
                             │                                │
                             └──────────────┬─────────────────┘
                                            │
                                            ▼
                                   New hidden state (h_t)
```

---

## 3. Mathematical Formulation

### Step 1 — Concatenate the inputs

The current input and previous hidden state are concatenated:

$$
z_t = [h_{t-1};x_t]
$$

This combined vector is used to compute the LSTM gates.

### Step 2 — Forget Gate

The forget gate decides how much information from the previous cell state should be retained.

$$
\boxed{f_t = \sigma(W_f z_t + b_f)}
$$

Where:

- $W_f$: Forget-gate weight matrix.
- $b_f$: Forget-gate bias.
- $\sigma$: Sigmoid activation function.

The sigmoid produces values between 0 and 1.

- Values near 0 indicate that information should largely be forgotten.
- Values near 1 indicate that information should largely be retained.

The retained memory is:

$$
f_t \odot c_{t-1}
$$

Here, $\odot$ represents element-wise multiplication.

### Step 3 — Input Gate

The input gate controls how much new information should be written into memory.

$$
\boxed{i_t = \sigma(W_i z_t + b_i)}
$$

A candidate cell state is also calculated:

$$
\boxed{\tilde{c}_t = \tanh(W_c z_t + b_c)}
$$

The candidate contains possible new memory values between -1 and 1.

The selected new information is:

$$
i_t \odot \tilde{c}_t
$$

### Step 4 — Update the Cell State

The new cell state combines retained previous information and selected new information:

$$
\boxed{
c_t = f_t \odot c_{t-1}
+ i_t \odot \tilde{c}_t
}
$$

This is the central equation of the LSTM.

It allows the network to:

1. Retain important information from previous time steps.
2. Forget information that is no longer useful.
3. Add relevant new information.

### Step 5 — Output Gate

The output gate controls which information from the updated cell state is exposed through the hidden state.

$$
\boxed{o_t = \sigma(W_o z_t + b_o)}
$$

The new hidden state is:

$$
\boxed{h_t = o_t \odot \tanh(c_t)}
$$

The hidden state can be used for predictions and is passed to the next time step, together with the cell state.

---

## 4. Why LSTM Helps With Vanishing Gradients

In a simple RNN, information propagates through repeated transformations:

$$
h_t = \tanh(W_h h_{t-1} + W_x x_t + b)
$$

During backpropagation, the chain rule introduces repeated multiplication by derivatives.

If these products become very small, gradients vanish.

An LSTM instead uses an additive cell-state update:

$$
c_t = f_t \odot c_{t-1}
+ i_t \odot \tilde{c}_t
$$

Along the direct cell-state path, treating the gates as fixed:

$$
\frac{\partial c_t}{\partial c_{t-1}} = f_t
$$

Over multiple time steps, this direct-path gradient is multiplied by the forget gates:

$$
\prod_{k=2}^{T} f_k
$$

If the forget-gate values remain close to 1, gradients can propagate through many time steps without rapidly disappearing along this path.

This is one reason LSTMs can learn longer-term dependencies than standard RNNs.

**Important:** LSTMs reduce the vanishing-gradient problem but do not eliminate it entirely. The complete gradient also includes paths through the gates, and exploding gradients can still occur.

---

## 5. Building LSTM From Scratch in PyTorch

In this project, we implement the LSTM equations manually using `torch` and `torch.nn`.

We deliberately avoid `nn.LSTM` so that the internal operations remain visible.

The four learnable transformations are:

```python
self.forget = nn.Linear(input_size + hidden_size, hidden_size)
self.input = nn.Linear(input_size + hidden_size, hidden_size)
self.candidate = nn.Linear(input_size + hidden_size, hidden_size)
self.output = nn.Linear(input_size + hidden_size, hidden_size)
```

At each time step, the network performs:

```python
z = torch.cat([h, x_t], dim=1)

f = torch.sigmoid(self.forget(z))
i = torch.sigmoid(self.input(z))
g = torch.tanh(self.candidate(z))
o = torch.sigmoid(self.output(z))

c = f * c + i * g
h = o * torch.tanh(c)
```

The same learnable weights are shared across all time steps.

PyTorch automatically tracks these operations, allowing gradients to be computed using Backpropagation Through Time.

---

## 6. Dataset and Learning Objective

We train our LSTM on a synthetic sequence-classification task.

**Objective:** Determine whether the digit `9` appears anywhere in an input sequence.

Each sample contains 10 integers between 0 and 9.

Examples:

| Input Sequence | Target |
|---|---|
| [2, 4, 9, 1, 5, 0, 7, 3, 1, 2] | 1 |
| [2, 4, 8, 1, 5, 0, 7, 3, 1, 2] | 0 |
| [1, 3, 2, 6, 4, 9, 5, 7, 8, 0] | 1 |

The model processes each sequence one element at a time and produces a binary prediction using its final hidden state.

The architecture is:

$$
\text{Input Sequence}
\rightarrow \text{Embedding}
\rightarrow \text{LSTM}
\rightarrow \text{Linear}
\rightarrow \text{Prediction}
$$

### Model Configuration

| Component | Value |
|---|---|
| Vocabulary size | 10 |
| Sequence length | 10 |
| Embedding dimension | 8 |
| Hidden-state dimension | 16 |
| Output dimension | 1 |
| Training samples | 2,000 |
| Test samples | 400 |
| Loss function | BCEWithLogitsLoss |
| Optimizer | Adam |
| Learning rate | 0.01 |
| Batch size | 64 |
| Epochs | 15 |

### Training Objective

The final linear layer produces a logit:

$$
a = W_y h_T+b_y
$$

We convert the logit into a probability:

$$
p = \sigma(a)
$$

Binary cross-entropy measures the prediction error:

$$
\mathcal{L}
=
-\left[
y\log(p)+(1-y)\log(1-p)
\right]
$$

In PyTorch, `BCEWithLogitsLoss` combines the sigmoid operation and binary cross-entropy in a numerically stable implementation.

During training, we also apply gradient clipping:

```python
nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

This helps control exploding gradients.

---

## 7. Key Takeaways

- LSTM is a gated recurrent architecture designed to learn dependencies across sequences.
- It maintains both a hidden state and a cell state.
- The forget gate controls memory retention.
- The input gate controls memory updates.
- The candidate state proposes new information.
- The output gate controls the exposed hidden representation.
- The cell-state update is additive, helping gradients propagate over longer distances.
- The same LSTM weights are reused across time steps.
- The complete sequence is trained using Backpropagation Through Time.
- In this implementation, the final hidden state is used for binary classification.

**Core LSTM equations:**

$$
\boxed{
\begin{aligned}
z_t &= [h_{t-1};x_t] \\
f_t &= \sigma(W_f z_t+b_f) \\
i_t &= \sigma(W_i z_t+b_i) \\
\tilde{c}_t &= \tanh(W_c z_t+b_c) \\
o_t &= \sigma(W_o z_t+b_o) \\
c_t &= f_t\odot c_{t-1}+i_t\odot\tilde{c}_t \\
h_t &= o_t\odot\tanh(c_t)
\end{aligned}
}
$$


In [2]:
import torch
import torch.nn as nn

first we create our dataset

In [4]:
def make_dataset(n, length = 10):
  X = torch.randint(0, 9, (n, length)) # generate number from 0 to 8 (not 9!)
  y = torch.randint(0, 2, (n,)).float() # generate numbers with 0 and 1.

  for i in range(n):
    if y[i] == 1:
      pos = torch.randint(0, length, (1,)).item() # this is the positino
      X[i, pos] = 9

  return X, y

x_train, y_train = make_dataset(4000)
x_test, y_test = make_dataset(600)

now we build the model from scratch

In [6]:
class MyLSTM(nn.Module):
  def __init__(self, vocab_size = 10, embed_size = 8, hidden_size = 16):
    super().__init__()
    self.hidden_size = hidden_size
    self.embedding = nn.Embedding(vocab_size, embed_size)

    self.forget = nn.Linear(embed_size + hidden_size, hidden_size)
    self.input = nn.Linear(embed_size + hidden_size, hidden_size)
    self.candidate = nn.Linear(embed_size + hidden_size, hidden_size)
    self.output = nn.Linear(embed_size + hidden_size, hidden_size)

    # and the prediction layer
    self.output = nn.Linear(hidden_size, 1)

  def forward(self, X):
    X = self.embedding(X) # (64, 10, 8)
    B, T, _ = X.shape
    # 64 samples
    # 10 time steps per sample
    # 8 embedding features per each time steps
    h = torch.new_zeros(B, self.hidden_size)  # (64, 16)
    c = torch.new_zeros(B, self.hidden_size)  # (64, 16)
    # at the begining of each independent sequence, the LSTM hasn't processed any digits yet.
    # thats why we fill the h and c with zeros
    # new_zeros() creates zeros with the same device and floating point type similar to the embedded tensor.
    for t in range(T): # T == 10
      x_t = X[:, t, :]
      z = torch.cat([h, x_t], dim=1) # z = X + h
      # X.shape = (64, 8)  the current input
      # h.shape = (64, 16) information from previous step
      #  ===> z.shape = (64, 24)
      f = torch.sigmoid(self.forget(z))
      i = torch.sigmoid(self.input(z))
      g = torch.tanh(self.candidate(z))
      o = torch.sigmoid(self.output(z))
      c = f * c + i * g
      h = o * torch.tanh(c)
    return self.classifier(h).sqeeze(1)

Model, Loss and optimization

In [7]:
model = MyLSTM()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

now the training part

In [ ]:
epochs = 15
batch_size = 64
for epoch in range(epochs):
  model.train()
  for start in range(0, len(x_train), batch_size): # 0, 64, 128, 192, 256 , ....
    xb = x_train[start: start + batch_size]
    yb = y_train[start: start + batch_size]
